<a href="https://colab.research.google.com/github/irezendegarcia-prog/python_sis_26_3_periodo/blob/Italo_Rezende/aula_14/Italo_Rezende_aula_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Atividade Prática 01 - Aula 14: Monitoramento via WeatherAPI

In [ ]:
# ==============================================================================
# API escolhida: WeatherAPI.com (Realtime / Current Weather API)
# Documentação oficial: https://www.weatherapi.com/docs/
# Endpoint Current Weather: https://www.weatherapi.com/docs/realtime-api/
# ==============================================================================

# Etapa 1: Configuração da consulta, chave segura (Secrets/getpass) e lista de cidades
import getpass

import requests

# Recupera a API_KEY de forma segura: primeiro tenta o Secrets do Google Colab
# (nome sugerido: WEATHERAPI_KEY); caso não esteja configurada, solicita via getpass.
try:
    from google.colab import userdata

    API_KEY = userdata.get("WEATHERAPI_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    API_KEY = getpass.getpass("Digite sua chave da WeatherAPI (API_KEY): ").strip()

BASE_URL = "https://api.weatherapi.com/v1/current.json"

cidades = [
    "Curitiba, Brazil",
    "Sao Paulo, Brazil",
    "Rio de Janeiro, Brazil",
    "Belo Horizonte, Brazil",
    "Porto Alegre, Brazil",
    "Salvador, Brazil",
    "Recife, Brazil",
    "Fortaleza, Brazil",
    "Brasilia, Brazil",
    "Manaus, Brazil",
]

print(f"Configuração concluída! {len(cidades)} cidades brasileiras prontas para consulta.")


In [ ]:
# Etapa 2: Requisições em laço com timeout=10, raise_for_status() e tratamento de erros
respostas_validas = []

for cidade in cidades:
    parametros = {
        "key": API_KEY,
        "q": cidade,
        "lang": "pt",
        "aqi": "no",
    }

    try:
        resposta = requests.get(BASE_URL, params=parametros, timeout=10)
        print(f"[{cidade}] Status HTTP: {resposta.status_code}")
        resposta.raise_for_status()
        respostas_validas.append(resposta)
    except requests.exceptions.HTTPError as erro_http:
        try:
            detalhe = resposta.json().get("error", {}).get("message", str(erro_http))
        except ValueError:
            detalhe = str(erro_http)
        print(f"  -> [ERRO HTTP] Falha ao consultar '{cidade}': {detalhe}")
    except requests.exceptions.Timeout:
        print(f"  -> [ERRO TIMEOUT] Tempo limite (10s) excedido ao consultar '{cidade}'.")
    except requests.exceptions.ConnectionError:
        print(f"  -> [ERRO CONEXÃO] Falha de rede ao consultar '{cidade}'.")
    except requests.exceptions.RequestException as erro_geral:
        print(f"  -> [ERRO REQUISIÇÃO] Erro inesperado em '{cidade}': {erro_geral}")

print(f"\nTotal de requisições bem-sucedidas: {len(respostas_validas)}/{len(cidades)}")


In [ ]:
# Etapa 3: Conversão para dicionário (.json()), inspeção de chaves (.keys()) e exibição
resultados_api = []

for indice, resposta in enumerate(respostas_validas):
    dados = resposta.json()

    if indice == 0:
        print("=== Inspeção das chaves principais (.keys()) ===")
        print("Chaves raiz:", list(dados.keys()))
        print("Chaves em 'location':", list(dados.get("location", {}).keys()))
        print("Chaves em 'current':", list(dados.get("current", {}).keys()))
        print()

    resultados_api.append(dados)

print("=== Monitoramento Climático Atual ===")
for registro in resultados_api:
    cidade = registro["location"]["name"]
    temperatura = registro["current"]["temp_c"]
    condicao = registro["current"]["condition"]["text"]
    print(f"- {cidade}: {temperatura:.1f} °C | Condição: {condicao}")


## Atividade Prática 02 - Aula 14: Análise com Pandas e Visualização

In [ ]:
# Etapa 1: Conversão da lista de dicionários em DataFrame, seleção e renomeação
import matplotlib.pyplot as plt
import pandas as pd

df_bruto = pd.json_normalize(resultados_api)

colunas_mapeadas = {
    "location.name": "Cidade",
    "location.region": "Estado",
    "location.localtime": "Hora_Local",
    "current.last_updated": "Ultima_Atualizacao",
    "current.temp_c": "Temperatura_C",
    "current.feelslike_c": "Sensacao_Termica_C",
    "current.humidity": "Umidade_Pct",
    "current.wind_kph": "Vento_Kph",
    "current.condition.text": "Condicao",
}

df_clima = df_bruto[list(colunas_mapeadas.keys())].rename(columns=colunas_mapeadas)

print(f"DataFrame estruturado com {df_clima.shape[0]} linhas e {df_clima.shape[1]} colunas:")
print(df_clima.head())


In [ ]:
# Etapa 2: Remoção de duplicatas, tipagem (to_datetime / to_numeric) e resumo estatístico
qtd_antes = len(df_clima)
df_clima = df_clima.drop_duplicates(subset=["Cidade", "Ultima_Atualizacao"]).copy()
print(f"Duplicatas removidas: {qtd_antes - len(df_clima)}")

# Conversão de colunas de data/hora
df_clima["Hora_Local"] = pd.to_datetime(df_clima["Hora_Local"], errors="coerce")
df_clima["Ultima_Atualizacao"] = pd.to_datetime(
    df_clima["Ultima_Atualizacao"], errors="coerce"
)

# Conversão explícita de colunas numéricas
colunas_numericas = [
    "Temperatura_C",
    "Sensacao_Termica_C",
    "Umidade_Pct",
    "Vento_Kph",
]
for coluna in colunas_numericas:
    df_clima[coluna] = pd.to_numeric(df_clima[coluna], errors="coerce")

print("\n=== Tipos de dados após tratamento (.info()) ===")
df_clima.info()

print("\n=== Resumo estatístico (.describe()) ===")
print(df_clima[colunas_numericas].describe().round(2))


In [ ]:
# Etapa 3: Visualização com boas práticas e exportação do DataFrame tratado (.csv)
# Pergunta: Como as temperaturas atuais variam entre as cidades brasileiras monitoradas
# em relação à média geral?

df_ordenado = df_clima.sort_values(by="Temperatura_C", ascending=True)
media_temp = df_ordenado["Temperatura_C"].mean()

cores = [
    "#D95F02" if temp >= media_temp else "#2E86AB"
    for temp in df_ordenado["Temperatura_C"]
]

fig, ax = plt.subplots(figsize=(9.5, 5.5))
barras = ax.barh(
    df_ordenado["Cidade"],
    df_ordenado["Temperatura_C"],
    color=cores,
    edgecolor="#222222",
    linewidth=0.7,
)

ax.axvline(
    media_temp,
    color="#333333",
    linestyle="--",
    linewidth=1.3,
    label=f"Média geral ({media_temp:.1f} °C)",
)

ax.set_title(
    "Temperatura Atual nas Cidades Brasileiras Monitoradas (WeatherAPI)",
    fontsize=13,
    fontweight="bold",
    pad=14,
)
ax.set_xlabel("Temperatura (°C)", fontsize=11, labelpad=8)
ax.set_ylabel("Cidade", fontsize=11, labelpad=8)
ax.set_xlim(0, df_ordenado["Temperatura_C"].max() * 1.18)
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for barra in barras:
    largura = barra.get_width()
    ax.text(
        largura + 0.4,
        barra.get_y() + barra.get_height() / 2,
        f"{largura:.1f} °C",
        va="center",
        ha="left",
        fontsize=9.5,
        fontweight="bold",
        color="#222222",
    )

ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()

arquivo_saida = "monitoramento_clima_tratado.csv"
df_clima.to_csv(arquivo_saida, index=False, encoding="utf-8")
print(f"Relatório tratado exportado com sucesso para '{arquivo_saida}'.")
